# Obese liver after a glucose load: a 19-gene metabolic panel

**Question.** Which reactions change in fasting obese liver, and through
which regulatory mechanism? What can a trans-omic analysis say about a small
targeted panel that separate lists of changed molecules cannot?

**Data.** [Uematsu *et al.*, *iScience* 25(2):103787, 2022](https://doi.org/10.1016/j.isci.2022.103787)
measured the liver transcriptome, proteome and metabolome in the same mice:
wild-type and leptin-deficient obese (ob/ob) animals, fasted and 4 hours
after an oral glucose load. The panel covers central carbon metabolism: 19
enzyme genes, their proteins, and 32 metabolites.

The data files are downloaded when the notebook runs.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from transnet import (
    convergence_significance,
    cross_layer_connectivity,
    downstream_influence,
    expression_concordance,
    layer_coverage,
    map_omics_to_network,
    metabolite_regulatory_roles,
    path_consistency_summary,
    reaction_regulation_table,
    regulation_axis_summary,
    regulatory_motifs,
    regulatory_role_enrichment,
    responsive_subnetwork,
    structural_vulnerability,
    trace_regulatory_paths,
    transcription_factor_activity,
    transomic_hubs,
)
from transnet.analysis import versus_chance
from transnet.api import kegg_reaction_pathways
from transnet.datasets import (
    DATA_DIR,
    UEMATSU_CONTRASTS,
    UEMATSU_LOG2FC,
    UEMATSU_METABOLITE_IDS,
    fetch_uematsu_panel,
    load_uematsu_panel,
    uematsu_contrast,
    uematsu_panel_network,
)
from transnet.io import to_arena3d, to_cytoscape_json, to_transomics2cytoscape, write_network
from transnet.visualization import (
    plot_axis_composition,
    plot_controversial_reactions,
    plot_convergence_null,
    plot_downstream_influence,
    plot_expression_concordance,
    plot_layer_connectivity,
    plot_metabolite_regulators,
    plot_regulation_axes,
    plot_regulatory_motifs,
    plot_regulatory_paths,
    plot_structural_vulnerability,
    plot_transomic_hubs,
    plot_transomic_network,
    plot_transomic_network_interactive,
)

OUT = DATA_DIR / "published_results" / "obese_liver"
OUT.mkdir(parents=True, exist_ok=True)
QVALUE = 0.1                      # the threshold the Kuroda laboratory uses


def save(figure, name):
    figure.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


fetch_uematsu_panel()
graph, id_map = uematsu_panel_network()
print(f"panel neighbourhood: {graph.number_of_nodes()} molecules, "
      f"{graph.number_of_edges()} relationships")

panel neighbourhood: 221 molecules, 387 relationships


The network is the part of the organism-wide mouse network around the panel:
its genes, their proteins, the reactions those enzymes catalyse, and every
metabolite on or regulating those reactions. With no protein interactions
among 19 enzymes, every edge connects two different layers.

In [2]:
connectivity = cross_layer_connectivity(graph)
print(f"{connectivity['cross_layer_fraction']:.0%} of the panel's edges cross layers")
save(plot_layer_connectivity(connectivity, title="Obese liver panel: edges between layers"),
     "layer_connectivity")

100% of the panel's edges cross layers


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
panels = {layer: load_uematsu_panel(layer) for layer in
          ("Transcriptome", "Proteome", "Metabolome")}
pd.DataFrame({layer: {"features": values.shape[0], "samples": values.shape[1]}
              for layer, (values, _) in panels.items()}).T

,features,samples
Transcriptome,19,47
Proteome,19,47
Metabolome,32,47


## What a per-layer analysis can say

Three lists of changed molecules, and the overlap between two of them. A
molecule counts as changed at 1.5-fold and q <= 0.1, the definition the
authors use. The main contrast is obese against lean liver, fasted.

In [4]:
LABEL, REFERENCE, TEST = UEMATSU_CONTRASTS[2]     # ob/ob vs WT, fasting
tables = {layer: uematsu_contrast(values, design, REFERENCE, TEST)
          for layer, (values, design) in panels.items()}

changed = {layer: table[(table["padj"] <= QVALUE)
                        & (table["log2FC"].abs() >= UEMATSU_LOG2FC)]
           for layer, table in tables.items()}
print(LABEL)
for layer, table in changed.items():
    print(f"  {layer:<15}{len(table):>3} changed of {len(tables[layer]):>3} measured")

shared = set(changed["Transcriptome"]["feature"]) & set(changed["Proteome"]["feature"])
print(f"  shared between transcript and protein lists: {len(shared)}")

ob/ob vs WT, fasting
  Transcriptome    3 changed of  19 measured
  Proteome         8 changed of  17 measured
  Metabolome       6 changed of  32 measured
  shared between transcript and protein lists: 2


/Users/krv114/Documents/GitHub/transnet/transnet/datasets.py:694: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  pvalue = stats.ttest_ind(b, a, axis=1, equal_var=False, nan_policy="omit").pvalue


That is all the separate lists can say. They cannot say which reactions are
affected, through which mechanism, or whether the layers agree. The rest of
this notebook answers those questions on the network.

## Mapping the data onto the network

In [5]:
extra = {name: ids for name, ids in UEMATSU_METABOLITE_IDS.items() if len(ids) > 1}

report = map_omics_to_network(
    graph, tables, id_column="feature", log2fc_column="log2FC", qvalue_column="padj",
    se_column="se", id_map=id_map, qvalue_threshold=QVALUE,
    log2fc_threshold=UEMATSU_LOG2FC,
)
# Mass spectrometry measures a sugar phosphate as one pool, while KEGG lists its
# anomers separately, so the measurement is attached to every form a reaction
# uses.
for name, ids in extra.items():
    row = tables["Metabolome"][tables["Metabolome"]["feature"] == name]
    if row.empty:
        continue
    for identifier in ids[1:]:
        if identifier in graph:
            graph.nodes[identifier].update(
                log2fc=float(row["log2FC"].iloc[0]), qvalue=float(row["padj"].iloc[0]),
                measured=True,
                regulated=int(np.sign(row["log2FC"].iloc[0]))
                if (row["padj"].iloc[0] <= QVALUE
                    and abs(row["log2FC"].iloc[0]) >= UEMATSU_LOG2FC) else 0,
            )
report.per_layer

,layer,n_supplied,n_matched,match_fraction,n_layer_nodes,layer_covered,n_up,n_down,n_undirected,note
0,Transcriptome,19,19,1.000,19,1.000000,2,1,0,
1,Proteome,17,17,1.000,19,0.894737,7,1,0,
2,Metabolome,32,28,0.875,125,0.224000,4,1,0,


`layer_coverage` shows the limit of this panel: nearly every transcript and
protein is measured, but only a quarter of the metabolites that the panel's
reactions touch.

In [6]:
layer_coverage(graph)

,layer,n_nodes,n_measured,measured_fraction,n_responsive,n_regulated,n_up,n_down,mean_degree
0,Transcriptome,19,19,1.000000,3,3,2,1,1.000000
1,Proteome,19,17,0.894737,8,8,7,1,4.473684
2,Reactions,58,0,0.000000,0,0,0,0,6.344828
3,Metabolome,125,32,0.256000,7,7,6,1,2.416000


## Which axis regulates each reaction

For every reaction, `reaction_regulation_table` asks whether it changed
through the amount of its enzyme (the enzyme or gene axis), through its
substrates, products and allosteric regulators (the metabolite axis), or both,
and whether the two axes agree.

In [7]:
regulation = reaction_regulation_table(graph)
regulated = regulation[(regulation["gene_axis"] != 0) | (regulation["metabolite_axis"] != 0)]
enzyme_only = regulated[(regulated["gene_axis"] != 0) & (regulated["metabolite_axis"] == 0)]
metabolite_only = regulated[(regulated["gene_axis"] == 0) & (regulated["metabolite_axis"] != 0)]
both = regulated[(regulated["gene_axis"] != 0) & (regulated["metabolite_axis"] != 0)]
controversial = regulated[regulated["controversial"]]

print(f"{len(regulated)} of {len(regulation)} reactions regulated: {len(enzyme_only)} through "
      f"enzyme amount only, {len(metabolite_only)} through metabolites only, "
      f"{len(both)} through both")
print(f"{len(controversial)} controversial (the two axes point opposite ways)")
regulation.to_csv(OUT / "reaction_regulation.csv", index=False)
regulated.head(8)[["reaction", "name", "gene_axis", "metabolite_axis", "gene_axis_evidence"]]

49 of 58 reactions regulated: 31 through enzyme amount only, 5 through metabolites only, 13 through both
9 controversial (the two axes point opposite ways)


,reaction,name,gene_axis,metabolite_axis,gene_axis_evidence
0,R00200,ATP:pyruvate 2-O-phosphotransferase; ATP + Pyr...,1,-1,protein
2,R00258,L-alanine:2-oxoglutarate aminotransferase; L-A...,0,1,None
6,R00430,GTP:pyruvate 2-O-phosphotransferase; GTP + Pyr...,1,-1,protein
7,R00431,GTP:oxaloacetate carboxy-lyase (adding GTP;pho...,-1,0,protein
8,R00572,CTP:pyruvate 2-O-phosphotransferase; CTP + Pyr...,1,-1,protein
9,R00658,2-phospho-D-glycerate hydro-lyase (phosphoenol...,1,0,protein
10,R00659,UTP:pyruvate 2-O-phosphotransferase; UTP + Pyr...,1,-1,protein
11,R00703,(S)-lactate:NAD+ oxidoreductase; (S)-Lactate +...,1,-1,protein


`gene_axis_evidence` shows what each enzyme-axis call rests on. With both
transcript and protein measured for almost every enzyme, most calls are
supported by both layers (`gene_protein`).

In [8]:
regulated["gene_axis_evidence"].value_counts(dropna=False).to_frame("reactions")

,reactions
gene_axis_evidence,
protein,43
None,5
gene_protein,1


### The controversial reactions

Each bar is one measured molecule's push on the reaction: to the right it
speeds the reaction up, to the left it slows it down. Yellow bars are the
enzyme axis, pink bars the metabolite axis.

In [9]:
controversial[["reaction", "name", "gene_axis", "metabolite_axis", "allosteric_regulators"]]

,reaction,name,gene_axis,metabolite_axis,allosteric_regulators
0,R00200,ATP:pyruvate 2-O-phosphotransferase; ATP + Pyr...,1,-1,C00041(inhibited)
6,R00430,GTP:pyruvate 2-O-phosphotransferase; GTP + Pyr...,1,-1,C00041(inhibited)
8,R00572,CTP:pyruvate 2-O-phosphotransferase; CTP + Pyr...,1,-1,C00041(inhibited)
10,R00659,UTP:pyruvate 2-O-phosphotransferase; UTP + Pyr...,1,-1,C00041(inhibited)
11,R00703,(S)-lactate:NAD+ oxidoreductase; (S)-Lactate +...,1,-1,
12,R00724,ITP:pyruvate 2-O-phosphotransferase; ITP + Pyr...,1,-1,C00041(inhibited)
21,R01138,dATP:pyruvate 2-O-phosphotransferase; dATP + P...,1,-1,C00041(inhibited)
27,R01858,dGTP:pyruvate 2-O-phosphotransferase; dGTP + P...,1,-1,C00041(inhibited)
29,R02320,NTP:pyruvate O2-phosphotransferase; Nucleoside...,1,-1,C00041(inhibited)


In [10]:
save(plot_controversial_reactions(graph, regulation), "controversial_reactions")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Per-pathway balance

`kegg_reaction_pathways` assigns each reaction to its KEGG pathways (mouse
pathways only, overview maps excluded). `regulation_axis_summary` then counts,
per pathway, how many reactions each axis activates or inhibits, and
`plot_regulation_axes` draws it. Reactions that KEGG places in no mouse
pathway are counted as `unassigned` and left out of the figure.

In [11]:
pathways = kegg_reaction_pathways(regulation["reaction"], organism="mmu")
pathway_balance = regulation_axis_summary(regulation, pathway_map=pathways)
pathway_balance.head(10)[["pathway", "n_reactions", "gene_activated", "gene_inhibited",
                          "metabolite_activated", "metabolite_inhibited", "n_controversial"]]

,pathway,n_reactions,gene_activated,gene_inhibited,metabolite_activated,metabolite_inhibited,n_controversial
0,unassigned,19,18,0,3,6,6
1,Glycolysis / Gluconeogenesis,12,9,2,1,3,2
2,Pyruvate metabolism,7,3,3,1,3,3
3,Biosynthesis of various nucleotide sugars,4,4,0,0,0,0
4,Citrate cycle (TCA cycle),4,0,3,1,0,0
5,Pentose phosphate pathway,3,3,0,1,0,0
6,"Neomycin, kanamycin and gentamicin biosynthesis",3,3,0,0,0,0
7,Starch and sucrose metabolism,2,1,0,1,1,0
8,Fructose and mannose metabolism,2,2,0,0,0,0
9,Amino sugar and nucleotide sugar metabolism,2,1,0,1,1,0


In [12]:
assigned = pathway_balance[(pathway_balance["pathway"] != "unassigned")
                           & (pathway_balance["n_reactions"] >= 2)]
save(plot_regulation_axes(assigned,
                          title="Obese against lean liver, fasted: regulation by pathway"),
     "regulation_axes")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Do the changed metabolites regulate enzymes?

A changed metabolite may simply follow the change in flux, or it may itself
act on an enzyme. `metabolite_regulatory_roles` lists, for each metabolite,
the reactions it activates or inhibits according to BRENDA, and
`regulatory_role_enrichment` tests whether changed metabolites are
regulators more often than measured metabolites in general.

In [13]:
roles = regulatory_role_enrichment(metabolite_regulatory_roles(graph))
counts = roles["counts"]
print(f"{counts['n_differential_regulators']} of {counts['n_differential']} changed metabolites "
      f"regulate an enzyme ({counts['fraction_differential_regulators']:.0%}, against "
      f"{counts['fraction_background_regulators']:.0%} of all measured metabolites)")
roles["regulators"][["name", "log2fc", "reactions_activated", "reactions_inhibited"]]

1 of 7 changed metabolites regulate an enzyme (14%, against 25% of all measured metabolites)


,name,log2fc,reactions_activated,reactions_inhibited
0,L-Alanine; L-2-Aminopropionic acid; L-alpha-Al...,0.730744,,R00200;R00430;R00572;R00659;R00724;R01138;R018...


In [14]:
save(plot_metabolite_regulators(roles), "metabolite_regulators")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Are the protein changes transcriptional?

Most enzyme-axis reactions change through the amount of enzyme. Is that
change transcriptional? For every gene measured in both layers,
`expression_concordance` asks whether the protein
followed its transcript. Because standard errors were mapped, it also tests
directly whether the protein changed *more* than its transcript, which does
not depend on where a significance cut-off falls.

In [15]:
concordance = expression_concordance(graph)
concordance_counts = concordance["counts"]
print(f"{concordance_counts['concordant']} concordant, "
      f"{concordance_counts['protein_only']} protein-only, "
      f"{concordance_counts['discordant']} against their transcript")
print(f"{concordance_counts.get('protein_beyond_transcript')} proteins moved significantly "
      f"further than their transcript, of {concordance_counts.get('tested_difference')} testable")

table = concordance["table"]
table[table.get("protein_beyond_transcript", False) == True][  # noqa: E712
    [c for c in ("name", "gene_log2fc", "protein_log2fc", "difference", "difference_q")
     if c in table.columns]
]

2 concordant, 6 protein-only, 0 against their transcript
4 proteins moved significantly further than their transcript, of 17 testable


,name,gene_log2fc,protein_log2fc,difference,difference_q
2,Ldha,0.303369,0.782984,0.479615,6.833318e-05
3,Eno1,0.404367,0.960049,0.555681,2.241710e-09
4,Gpi1,0.196467,1.203885,1.007418,9.067268e-14
5,Fbp1,-0.140187,1.196494,1.336681,2.338090e-56


In [16]:
save(plot_expression_concordance(concordance), "concordance")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Transcription factors, and the molecules that connect layers

The panel network has no transcription-factor edges that reach its 19
genes, so the factor inference cannot run here; this is a limit of the panel,
not a finding. The liver time-course study runs it genome-wide. The
cross-layer hubs are informative: they are the molecules through which the
response in one layer reaches another.

In [17]:
factors = transcription_factor_activity(graph, min_targets=3)
implicated = factors[factors["q_value"] <= 0.05] if not factors.empty else factors
print(f"{len(factors)} transcription factors testable, {len(implicated)} implicated")

hubs = transomic_hubs(responsive_subnetwork(graph), top_percent=10)
hubs.head(8)[["name", "layer", "cross_layer_degree", "n_layers_touched"]]

No transcriptional_regulation edges reach a measured gene; transcription factor activity cannot be inferred


0 transcription factors testable, 0 implicated


,name,layer,cross_layer_degree,n_layers_touched
0,Pyruvate kinase PKLR (EC 2.7.1.40) (L-PK) (Pyr...,Proteome,9,2
1,L-Alanine; L-2-Aminopropionic acid; L-alpha-Al...,Metabolome,8,1
2,Glucose-6-phosphate isomerase (GPI) (EC 5.3.1....,Proteome,5,1
3,alpha-D-Glucose 6-phosphate,Metabolome,3,1
4,beta-D-Glucose 6-phosphate,Metabolome,2,1
5,D-Glucose 6-phosphate; Glucose 6-phosphate; Ro...,Metabolome,1,1
6,ATP; Adenosine 5'-triphosphate,Metabolome,1,1
7,Pklr,Transcriptome,1,1


In [18]:
save(plot_transomic_hubs(hubs, title="Obese liver: molecules connecting layers"),
     "transomic_hubs")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Do the enzyme changes predict the metabolite changes?

If the network is right, the changes in transcripts and proteins should
predict which way the metabolites moved. Two methods ask this (see the
*signed regulatory paths* walkthrough). Propagation pushes the enzyme changes
forward along signed edges and gives each metabolite a predicted direction.

In [19]:
seeds = {n: float(d["log2fc"]) for n, d in graph.nodes(data=True)
         if d.get("layer") in ("Transcriptome", "Proteome")
         and d.get("regulated") and d.get("log2fc") is not None}
influence = downstream_influence(graph, seeds, target_layer="Metabolome")
measured = influence[influence["observed"].fillna(0) != 0]
if not measured.empty:
    agree = int(measured["agrees"].astype("boolean").fillna(False).sum())
    print(f"{agree} of {len(measured)} changed metabolites predicted correctly: "
          f"{versus_chance(agree, len(measured))}")
measured.head(10)[["name", "score", "predicted_direction", "observed", "agrees"]]

1 of 1 changed metabolites predicted correctly: 100%, no better than the 50% expected by chance (binomial p = 1)


,name,score,predicted_direction,observed,agrees
29,beta-D-Glucose 6-phosphate,0.006213,1,1,True


In [20]:
save(plot_downstream_influence(influence), "downstream_influence")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Path tracing lists the individual routes from each changed gene or protein to
each changed metabolite, and checks whether the sign along each route matches
the measured direction. `path_consistency_summary` gives one verdict per
metabolite.

In [21]:
paths = trace_regulatory_paths(graph, target_layer="Metabolome", max_length=4)
summary = path_consistency_summary(paths)
if not summary.empty:
    agree = int(summary["agrees"].astype("boolean").fillna(False).sum())
    print(f"{len(paths)} signed paths to {len(summary)} changed metabolites; "
          f"{agree} predicted in the right direction: "
          f"{versus_chance(agree, len(summary))}")
else:
    print(f"{len(paths)} signed paths, none reaching a metabolite that changed")
summary.head(10)

1 signed paths to 1 changed metabolites; 1 predicted in the right direction: 100%, no better than the 50% expected by chance (binomial p = 1)


,target,observed,n_paths,n_fully_signed,n_consistent,fraction_consistent,predicted,agrees,shortest_consistent_path
0,C01172,1,1,1,1,1.0,1,True,P06745 -> R02739 -> C01172


In [22]:
save(plot_regulatory_paths(paths, graph, top_n=8), "regulatory_paths")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recurring wiring patterns, weak points, and a null model

`regulatory_motifs` searches the signed wiring among the changed molecules
for product inhibition, allosteric feedback and feed-forward patterns, which
can explain a controversial reaction rather than just flag it. On this panel
it finds none: the one allosteric regulator that changed, alanine, acts on
pyruvate kinase, which does not produce it.

In [23]:
motifs = regulatory_motifs(graph)
print(motifs["counts"])
save(plot_regulatory_motifs(motifs), "regulatory_motifs")
motifs["motifs"][["motif", "reaction_name", "metabolite_name", "enzyme", "sign_product"]].head(10)

{}


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,motif,reaction_name,metabolite_name,enzyme,sign_product


Is the number of reactions where both axes changed more than chance would
produce? `convergence_significance` reassigns at random which molecules
changed, keeping the number per layer fixed, and counts again.

In [24]:
convergence = convergence_significance(graph, n_randomisations=500)
print(f"{convergence['observed']} reactions with both axes changed; "
      f"{convergence['null_mean']:.1f} expected by chance "
      f"(z = {convergence['z']:+.1f}, p = {convergence['p_value']:.3g})")
save(plot_convergence_null(convergence), "convergence_null")

14 reactions with both axes changed; 6.3 expected by chance (z = +1.9, p = 0.0499)


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`structural_vulnerability` lists the molecules whose removal would split the
responsive network into separate pieces.

In [25]:
vulnerability = structural_vulnerability(responsive_subnetwork(graph), top_n=8)
save(plot_structural_vulnerability(vulnerability), "structural_vulnerability")
vulnerability

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


""


## Are the layers from the same animals?

A joint factor model assumes that column *i* is the same mouse in every file.
The authors describe the layers as measured in the same individuals, and the
data can check this: within a group, a mouse whose transcript of a gene is
high should also tend to have more of that protein, and this agreement
should be stronger in the listed pairing than in shuffled ones.

In [26]:
from transnet.analysis.factors import (
    factor_design_association,
    factor_network_coherence,
    factor_network_propagation,
    fit_factors,
    pairing_robustness,
    sample_pairing_check,
)

transcript_values, transcript_design = panels["Transcriptome"]
protein_values, _ = panels["Proteome"]
genes = sorted(set(transcript_values.index) & set(protein_values.index))
groups = (transcript_design["genotype"] + " " + transcript_design["minutes"]).astype(str)
groups.index = transcript_values.columns

pairing = sample_pairing_check(transcript_values.loc[genes].T, protein_values.loc[genes].T,
                               groups, n_permutations=500)
print(f"agreement as listed {pairing['observed']:.3f}, under shuffled pairings "
      f"{pairing['null_mean']:.3f} (permutation p = {pairing['p_value']:.2f}) -> "
      f"{'paired' if pairing['paired'] else 'pairing not confirmed'}")

/Users/krv114/opt/anaconda3/envs/transnet/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/krv114/opt/anaconda3/envs/transnet/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/krv114/opt/anaconda3/envs/transnet/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/krv114/opt/anaconda3/envs/transnet/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


agreement as listed 0.066, under shuffled pairings -0.004 (permutation p = 0.10) -> pairing not confirmed


With only 17 genes in both layers the check has little power: the agreement
points the right way but is not significant, so the pairing is neither
confirmed nor refuted.

The factor model is therefore fitted and then tested for whether the pairing
matters. `pairing_robustness` shuffles the mice within each group and checks
whether each factor keeps its agreement across layers. A factor marked
`between-group` is carried by genotype or glucose, whose labels are certain,
and can be interpreted whatever the pairing. A factor that depends on
mouse-to-mouse variation is not interpreted until the pairing is confirmed.

In [27]:
matrices = {}
for layer, (values, _) in panels.items():
    frame = values.T.apply(pd.to_numeric, errors="coerce")
    frame = np.log2(frame.clip(lower=0) + (1.0 if np.nanmax(frame.to_numpy()) > 50 else 1e-3))
    matrices[layer] = frame.loc[:, frame.notna().mean() >= 0.8]
design = transcript_design.assign(sample=transcript_values.columns).set_index("sample")[
    ["genotype", "minutes"]].astype(str)

factorisation = fit_factors(matrices, n_components=4)
association = factor_design_association(factorisation.factors_, design, ["genotype", "minutes"])
robustness = pairing_robustness(factorisation, groups, n_shuffles=200)
factor_ids = {"Transcriptome": id_map["Transcriptome"], "Proteome": id_map["Proteome"],
              "Metabolome": id_map["Metabolome"]}
coherence = factor_network_coherence(graph, factorisation.loadings_, id_maps=factor_ids,
                                     top_n=10, n_permutations=500)
propagation = factor_network_propagation(graph, factorisation.loadings_, id_maps=factor_ids,
                                         top_n=10, n_permutations=200)

(association.pivot(index="factor", columns="term", values="partial_eta_squared")
 .join(robustness.set_index("factor")[["agreement", "retained", "verdict"]])
 .join(coherence["table"].set_index("factor")[["fold_enrichment"]]
       .rename(columns={"fold_enrichment": "direct links vs chance"}))
 .join(propagation["table"].set_index("factor")[["overlap", "null_overlap", "q_value"]]
       .rename(columns={"q_value": "overlap q"}))
 .round(3))

/Users/krv114/opt/anaconda3/envs/transnet/lib/python3.10/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


,genotype,genotype x minutes,minutes,agreement,retained,verdict,direct links vs chance,overlap,null_overlap,overlap q
factor,,,,,,,,,,
Factor1,0.728,0.158,0.430,0.736,0.962,between-group,1.182,0.395,0.348,0.418
Factor2,0.455,0.182,0.251,0.271,0.671,single-layer,0.875,0.406,0.396,0.557
Factor3,0.678,0.295,0.235,-0.040,NaN,single-layer,0.874,0.298,0.350,0.811
Factor4,0.798,0.325,0.269,0.736,0.961,between-group,1.241,0.433,0.367,0.418


In the table, the genotype and minutes columns give the share of each factor's
variation explained by that part of the design. `direct links vs chance` says
how much more often the factor's top molecules are directly connected on the
network than random molecules, and `overlap` how much the network
neighbourhoods of its top molecules in different layers overlap.

In [28]:
from transnet.visualization import plot_factor_network, plot_factor_scores

FACTOR_OUT = OUT / "factors"
FACTOR_OUT.mkdir(parents=True, exist_ok=True)

design_plot = design.assign(group=design["genotype"] + " " + design["minutes"] + " min")
plot_factor_scores(factorisation.factors_, design_plot, x="group",
                   x_order=["WT 0 min", "WT 240 min", "ob 0 min", "ob 240 min"],
                   association=association,
                   title="Obese liver: factor scores across the design"
                   ).savefig(FACTOR_OUT / "factor_scores.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/2959065867.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
robust_factors = robustness.loc[robustness["verdict"] == "between-group", "factor"].tolist()
for factor in robust_factors:
    linked = coherence["nodes"].get(factor, [])
    figure = plot_factor_network(graph, factorisation.loadings_, factor,
                                 id_maps=factor_ids, nodes=linked or None,
                                 top_n=10, max_nodes=60,
                                 title=f"{factor}: the obesity signal, on the network")
    figure.savefig(FACTOR_OUT / f"network_{factor}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"{factor}: where its layers meet")
    print(propagation["top_nodes"][factor].head(8)[["name", "layer", "enrichment"]]
          .round(1).to_string(index=False))

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/2381360852.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Factor1: where its layers meet
                                            name      layer  enrichment
                                         Citrate Metabolome         5.5
                           D-Glucose 6-phosphate Metabolome         4.9
                     D-Fructose 1,6-bisphosphate Metabolome         3.4
               alpha-D-glucose 1,6-phosphomutase  Reactions         3.1
acetyl-CoA:oxaloacetate C-acetyltransferase (thi  Reactions         2.8
                                        Fumarate Metabolome         2.7
   D-glucose-6-phosphate aldose-ketose-isomerase  Reactions         2.5
                                            Gpd1   Proteome         2.2


Factor4: where its layers meet
                                         name         layer  enrichment
                       D-Fructose 6-phosphate    Metabolome         4.8
                        D-Glucose 6-phosphate    Metabolome         4.6
                                  L-Glutamate    Metabolome         3.4
                                  L-Aspartate    Metabolome         3.3
D-glucose-6-phosphate aldose-ketose-isomerase     Reactions         3.3
                                    Succinate    Metabolome         3.2
            alpha-D-glucose 1,6-phosphomutase     Reactions         3.2
                                         Pklr Transcriptome         2.5


/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/2381360852.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Two findings that no list of changed molecules contains

In [31]:
def describe(symbol_or_name):
    """Fold changes for one molecule, whatever layer it is in."""
    rows = []
    for node, data in graph.nodes(data=True):
        label = str(data.get("symbol") or data.get("name") or node)
        if label.split(";")[0].strip().lower() == symbol_or_name.lower():
            rows.append({"node": node, "layer": data.get("layer"), "name": label[:40],
                         "log2FC": data.get("log2fc"), "regulated": data.get("regulated")})
    return pd.DataFrame(rows)


pd.concat([describe(name) for name in ("Pklr", "L-Alanine", "Ldha", "Lactate")],
          ignore_index=True)

,node,layer,name,log2FC,regulated
0,18770,Transcriptome,Pklr,1.737997,1
1,P53657,Proteome,Pklr,1.998819,1
2,C00041,Metabolome,L-Alanine; L-2-Aminopropionic acid; L-al,0.730744,1
3,16828,Transcriptome,Ldha,0.303369,0
4,P06151,Proteome,Ldha,0.782984,1


**Pyruvate kinase is pushed in both directions.** Obese liver has more of the
enzyme, and also more alanine, its classic allosteric inhibitor in liver. The
extra enzyme is held back by the extra inhibitor. This is the regulation that
limits wasteful cycling between pyruvate and phosphoenolpyruvate while the
liver makes glucose. Alanine is found as a regulator only because the name
matching resolves BRENDA's spelling, "L-Ala".

**Lactate dehydrogenase rises while its substrate falls.** This is consistent
with lactate being used faster for glucose production, a hypothesis that a
flux measurement could test.

## The other two contrasts

The same analysis for the glucose response of lean and of obese liver.

In [32]:
rows = []
for label, reference, test in UEMATSU_CONTRASTS:
    other = graph.copy()
    contrast_tables = {layer: uematsu_contrast(values, design, reference, test)
                       for layer, (values, design) in panels.items()}
    map_omics_to_network(other, contrast_tables, id_column="feature", log2fc_column="log2FC",
                         qvalue_column="padj", se_column="se", id_map=id_map,
                         qvalue_threshold=QVALUE, log2fc_threshold=UEMATSU_LOG2FC)
    table = reaction_regulation_table(other)
    active = table[(table["gene_axis"] != 0) | (table["metabolite_axis"] != 0)]
    rows.append({
        "contrast": label,
        "regulated": len(active),
        "enzyme_axis_only": int(((active["gene_axis"] != 0) & (active["metabolite_axis"] == 0)).sum()),
        "metabolite_axis_only": int(((active["gene_axis"] == 0) & (active["metabolite_axis"] != 0)).sum()),
        "both_axes": int(((active["gene_axis"] != 0) & (active["metabolite_axis"] != 0)).sum()),
        "controversial": int(active["controversial"].sum()),
    })
composition = pd.DataFrame(rows)
composition

/Users/krv114/Documents/GitHub/transnet/transnet/datasets.py:694: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  pvalue = stats.ttest_ind(b, a, axis=1, equal_var=False, nan_policy="omit").pvalue
/Users/krv114/Documents/GitHub/transnet/transnet/datasets.py:694: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  pvalue = stats.ttest_ind(b, a, axis=1, equal_var=False, nan_policy="omit").pvalue
/Users/krv114/Documents/GitHub/transnet/transnet/datasets.py:694: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requir

,contrast,regulated,enzyme_axis_only,metabolite_axis_only,both_axes,controversial
0,WT glucose response,6,0,6,0,0
1,ob/ob glucose response,4,0,4,0,0
2,"ob/ob vs WT, fasting",49,31,5,13,9


Healthy liver responds to glucose through its metabolites; fasted obese liver
is changed through the amount of its enzymes.

In [33]:
save(plot_axis_composition(composition), "axes_by_contrast")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The responsive network

In [34]:
responsive = responsive_subnetwork(graph)
save(plot_transomic_network(responsive, title=LABEL), "network")

/var/folders/4t/z957yjm52ks8l2j2grrchnfm0000gn/T/ipykernel_45849/3055582069.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Exports

The responsive network in every format TransNet writes, for use in other
tools (see the *saving, exporting and sharing* walkthrough).

In [35]:
exports = OUT / "exports"
exports.mkdir(exist_ok=True)
write_network(responsive, str(exports / "csv"))
to_cytoscape_json(responsive, exports / "network_cytoscape.json")
to_arena3d(responsive, exports / "network_arena3d.json")
to_transomics2cytoscape(responsive, zip_path=exports / "network_transomics2cytoscape.zip")
plot_transomic_network_interactive(responsive, layout="layered", title=LABEL).write_html(
    exports / "network.html")
print(f"figures and tables in {OUT}")
print("exports:", ", ".join(sorted(p.name for p in exports.iterdir())))

figures and tables in /Users/krv114/Documents/GitHub/transnet/data/published_results/obese_liver
exports: csv, network.html, network_arena3d.json, network_cytoscape.json, network_transomics2cytoscape.zip
